# Self-Shaping WFS Mask — Learning the Sensor Optics from Scratch

This notebook trains a fully free-form, learned WFS phase mask **jointly** with a CNN reconstructor, and compares it against a classic modulated Pyramid WFS run two ways: with its own trained CNN reconstructor (the fair, same-compute-budget baseline) and with its existing frozen linear reconstructor (the reference floor). All three conditions share one telescope, atmosphere, DM, and noise model. See `Ideas/01-self-shaping-mask.md` for the full design discussion this notebook implements.

**The three conditions:**
- **A -- Self-shaping mask + CNN**: a generic `AI4AO.WFS` fitted with a single, fully unconstrained free-form phase mask (`AI4AO.MaskGeneration.FreeMaskGenerator`, `number_of_masks=1`), trained jointly with a CNN reconstructor that consumes the raw full detector frame (there is no known pupil-replica geometry for an arbitrary learned mask to crop against).
- **B -- Pyramid + CNN**: a standard modulated `AI4AO.PyramidWFS`, with its own CNN reconstructor trained on `AI4AO.FramePreprocess`-cropped pupil images.
- **C -- Pyramid + linear**: the *same* Pyramid WFS as B, using its classical linear reconstructor (`WFS.BuildReconstructionMatrix`/`GetReconstructedOPD`) instead of a CNN. Never trained -- the reference floor.

**Training scheme:** `AI4AO.Trainer.Trainer` cannot be reused here -- it knows nothing about a mask generator, and this repo's ablation convention is to share one atmosphere/noise draw per step across every compared condition rather than retraining each independently. So conditions A and B are trained together in one hand-rolled loop that draws the atmosphere once per step and runs both closed loops against it. Condition A's mask is only rebuilt (with gradients) every `MaskUpdatePeriod` steps -- `MaskUpdatePeriod = 1` makes *every* step a true joint mask+CNN update from one shared backward pass; larger values hold the mask fixed for several steps while the CNN adapts, genuine alternating/coordinate-descent training.

**New loss term:** `AI4AO.LossFunctions.CenterOfGravityLoss` penalizes the WFS frame's photocenter drifting away from the frame center -- a learned mask is otherwise free to add a global phase tilt that shifts the whole diffraction pattern without adding any useful signal, and nothing else in the loss discourages that. It replaces the least-squares tip/tilt removal `MaskGeneration.py` used to apply directly to the mask before that step was deleted. It is only added to condition A's loss; a fixed Pyramid mask is already frame-centered by construction.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display as ipy_display, clear_output
import numpy as np
import torch
import torch.nn as nn
import os
from tqdm import tqdm

from AI4AO import WFS, PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, imshow, imshow_multiple
from AI4AO.MaskGeneration import *
from AI4AO.Trainer import EvaluationResult
from AI4AO.LossFunctions import LogResidualVarianceLoss, WFSSignalLoss, Physics_loss, CenterOfGravityLoss
from AI4AO.paths import DATA_DIR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

## Configuration

`SelfShapingMask_params.py` reuses `WFSParams`/`AtmosParams`/`LoopParams`/`DMParams` verbatim from `ResidualCNNReconstructor_params.py` (same nominal modulated-Pyramid-scale telescope) so this notebook's results are directly comparable to that one's Pyramid conditions, rather than introducing yet another incomparable fictional instrument.

In [ ]:
paramfile = 'SelfShapingMask_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "SelfShapingMask"
os.makedirs(PATH, exist_ok=True)

## Shared instrument: DM and atmosphere

One `DeformableMirror` and one `PhaseDataset`, shared by every condition -- the physical mirror and the atmosphere statistics are identical across the comparison; only the WFS/mask and reconstructor differ. `dm`'s geometry/influence functions are fully deterministic given `WFSParams`/`DMParams` (no randomness anywhere in `DeformableMirror.__init__`), so there is no reason to build it three times.

In [ ]:
dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

M2C = dm.MakeZernikeM2C()
modes = dm(M2C.T)  # (Nmodes, Nres, Nres): the actual OPD the DM produces for each M2C column
M2C_T = M2C.T
z_inv = torch.linalg.pinv(modes.flatten(start_dim=-2))

Nmodes = DMParams['Nmodes']

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

## Conditions B and C: the classic modulated Pyramid

One `PyramidWFS`, shared by conditions B (its own trained CNN) and C (its calibrated linear reconstructor) -- both read the same fixed, hand-designed mask. `FramePreprocess` crops the 4 pupil images B's CNN consumes.

In [ ]:
wfs_pyramid = PyramidWFS(WFSParams, device)
wfs_pyramid.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs_pyramid, device)
framePreprocessor.ProcessReference(wfs_pyramid.reference_intensity)

n_channels_B = wfs_pyramid.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
print(f"Pyramid pupils: {n_channels_B}, preprocessed pupil size: {Nout}x{Nout}")

**Condition C's calibration.** `modes = dm(M2C.T)` (already computed above) is the same basis every condition's DM commands target, so calibrating the linear reconstructor against it (rather than a separately-built Zernike basis) keeps it exactly consistent with `M2C`. `wfs_pyramid` is frozen, so this only needs to happen once, before any training starts. Condition C has zero trainable parameters and is never touched by the training loop below -- only by the final comparison rollout.

In [ ]:
with torch.no_grad():
    wfs_pyramid.BuildReferenceIntensity()
    wfs_pyramid.BuildReconstructionMatrix(modes)

print(f"Linear reconstructor calibrated: reconstructionMatrix {tuple(wfs_pyramid.reconstructionMatrix.shape)}")

## Condition A: the self-shaping mask

`WFS` (the generic propagator, not `PyramidWFS`/`ZernikeWFS`) fitted with a `FreeMaskGenerator(number_of_masks=1)`: a single, near-flat, fully unconstrained `(N, N)` phase mask with no facet structure at all. `mask_gen(wfs_free)` builds the mask and calls `wfs_free.SetMask(...)` -- this **must** be called at least once before any forward pass, since `SetMask` only ever writes a cached `wfs_free.mask` that `forward()`/`Propagator()` read without ever rebuilding it on their own.

**Important:** unlike `PyramidWFS`/`ZernikeWFS`, plain `WFS` has no `BuildMask()` override, so calling `.eval()`/`.train()` on `wfs_free` would raise `AttributeError` (the base `WFS.train()` override unconditionally calls `self.BuildMask()`). `wfs_free` also has no trainable parameters of its own (only `mask_gen` does), so there is nothing to freeze on it anyway -- its mask is entirely driven by explicit `mask_gen(wfs_free)` calls in the training loop below.

In [ ]:
wfs_free = WFS(WFSParams, device)
mask_gen = ModalMaskGeneration(WFSParams, device=device).to(device=device)
mask_gen(wfs_free)  # first mask build -- required before any wfs_free forward call

total_params_mask = sum(p.numel() for p in mask_gen.parameters() if p.requires_grad)
print(f"Self-shaping mask generator -- total trainable parameters: {total_params_mask:,}")

**No `FramePreprocess` for condition A.** With `number_of_masks=1` there is no facet structure and no known pupil-replica location -- `wfs_free.pupil_centers` is never set (unlike `PyramidWFS.BuildMask`/`ZernikeWFS.BuildMask`, nothing in `FreeMaskGenerator` computes it, and it can't be derived from the mask geometry alone). Condition A's CNN instead consumes the raw full `(Npix, Npix)` detector frame directly. It still needs *some* normalization so its input sits on a comparable scale to condition B's `FramePreprocess`-normalized pupils, so we build a fixed reference normalization once, from a noiseless zero-OPD frame through the mask's current (initial) state -- mirroring `FramePreprocess.ProcessReference`'s own one-time calibration. This normalization does go slightly stale as the mask keeps evolving during training; that's an accepted approximation, not treated as exact.

In [ ]:
def normalize_frame(frame, normalization):
    return (frame - frame.mean(dim=(-2, -1), keepdim=True)) / normalization


with torch.no_grad():
    temp_noise = wfs_free.useNoise
    wfs_free.useNoise = False
    reference_frame_A = wfs_free(torch.zeros(1, WFSParams['Nres'], WFSParams['Nres'], device=device))
    wfs_free.useNoise = temp_noise
    normalization_A = torch.std(reference_frame_A, dim=(-2, -1), keepdim=True)

print(f"Condition A raw frame size: {tuple(reference_frame_A.shape[-2:])}, reference std: {float(normalization_A):.3e}")

## Sanity check

A quick visual check before building the training loop, since this is the first time this notebook exercises the free-form mask path end to end: the initial (near-flat, random) free-form mask and a sample frame through it, next to the Pyramid's fixed mask and a sample frame through it. The free-form mask's frame should look like a single diffuse blob (no pupil-splitting structure), quite different from the Pyramid's four distinct pupil images.

In [ ]:
sample_batch = dataset[0]
sample_opd = sample_batch["opd"]
sample_pupil = sample_batch["pupil"]
wfs_free.SetPhotonsAndRON(sample_batch["nphotons"], sample_batch["ron"])
wfs_pyramid.SetPhotonsAndRON(sample_batch["nphotons"], sample_batch["ron"])

with torch.no_grad():
    sample_frame_A = wfs_free(sample_opd, sample_pupil)
    sample_frame_B = wfs_pyramid(sample_opd, sample_pupil)

imshow_multiple(
    [
        {"tensor": mask_gen.phaseMask.detach(), "title": "Initial free-form mask (rad)", "colorbar": True},
        {"tensor": sample_frame_A, "title": "Sample frame through free-form mask", "colorbar": True},
        {"tensor": wfs_pyramid.phaseMask.detach(), "title": "Fixed Pyramid mask (rad)", "colorbar": True},
        {"tensor": sample_frame_B, "title": "Sample frame through Pyramid mask", "colorbar": True},
    ]
)
plt.show()

## Reconstructor architecture

One `PupilCNN` class (the stem/encoder/head shape used throughout `Tutorials/Advanced/` -- `ResidualCNNReconstructor.ipynb`, `CNNArchitectureComparison.ipynb`, `TwoStageAO.ipynb`'s `PWFSNet`), instantiated twice: `n_channels=1` for condition A's raw frame, `n_channels=4` for condition B's cropped Pyramid pupils. The stem's `groups=n_channels` is a no-op when `n_channels=1`, so the same class genuinely covers both cases -- no second architecture needed.

In [ ]:
class PupilCNN(nn.Module):
    def __init__(self, n_channels, Nmodes):
        super().__init__()

        self.stem = nn.Sequential(
            # Process each pupil image independently
            nn.Conv2d(n_channels, 8 * n_channels, kernel_size=11, padding=5, groups=n_channels),
            nn.GELU(),

            nn.Conv2d(8 * n_channels, 16 * n_channels, kernel_size=7, padding=3, groups=n_channels),
            nn.GELU(),

            nn.MaxPool2d(2),
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(16 * n_channels, 64, kernel_size=5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, Nmodes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

In [ ]:
cnn_A = PupilCNN(n_channels=1, Nmodes=Nmodes).to(device=device)
cnn_B = PupilCNN(n_channels=n_channels_B, Nmodes=Nmodes).to(device=device)

total_params_A = sum(p.numel() for p in cnn_A.parameters() if p.requires_grad)
total_params_B = sum(p.numel() for p in cnn_B.parameters() if p.requires_grad)
print(f"Condition A CNN -- total trainable parameters: {total_params_A:,}")
print(f"Condition B CNN -- total trainable parameters: {total_params_B:,}")

## Losses

`CenterOfGravityLoss` (new, added to `AI4AO/LossFunctions.py`) penalizes `wfs_frames`'s intensity-weighted centroid drifting from the frame center -- the direct replacement for the tip/tilt removal `MaskGeneration.py` used to apply to the mask itself before that step was deleted. It is a global, whole-frame centroid: a spurious *common* tip/tilt added on top of the mask shifts the whole frame's photocenter, which is exactly what this penalizes.

Condition A's loss is the equal-weight sum you specified: `LogResidualVarianceLoss + WFSSignalLoss + CenterOfGravityLoss` (`WFSSignalLoss` is already `-mean(std(wfs_frames)) * 1e6`, the same quantity as the `-torch.mean(wfs_frames.std(dim=(-2,-1)))` cell in `prototyping/MaskTests.ipynb`, just pre-scaled).

Condition B keeps this repo's usual `LogResidualVarianceLoss + Physics_loss` -- unchanged from `ResidualCNNReconstructor.ipynb`'s own convention. `CenterOfGravityLoss` is **not** added to B (a fixed Pyramid mask is already frame-centered, so it would be a permanent no-op), and `Physics_loss` is **not** added to A: `Physics_loss` deep-copies `wfs` once at construction and reuses that frozen copy's mask on every step, which is only valid for a mask that never changes -- exactly the opposite of what condition A's mask is doing.

In [ ]:
loss_A = LogResidualVarianceLoss(dataset.pupil, wfs_free.wavelength) + CenterOfGravityLoss(wfs_free) + WFSSignalLoss()
loss_B = LogResidualVarianceLoss(dataset.pupil, wfs_pyramid.wavelength)# + Physics_loss(wfs=wfs_pyramid)

## Optimizers

Three separate optimizers: the raw, unscaled mask phase (`mask_gen`), condition A's CNN weights, and condition B's CNN weights all sit at different scales.

In [ ]:
mask_optimizer = torch.optim.AdamW(mask_gen.parameters(), TrainParams['lr_mask'], fused=(device == 'cuda'))
cnn_A_optimizer = torch.optim.AdamW(cnn_A.parameters(), TrainParams['lr_cnn_A'], fused=(device == 'cuda'))
cnn_B_optimizer = torch.optim.AdamW(cnn_B.parameters(), TrainParams['lr_cnn_B'], fused=(device == 'cuda'))


def save_condition_A_checkpoint(path):
    torch.save({
        "mask_gen_state_dict": mask_gen.state_dict(),
        "cnn_A_state_dict": cnn_A.state_dict(),
        "mask_optimizer_state_dict": mask_optimizer.state_dict(),
        "cnn_A_optimizer_state_dict": cnn_A_optimizer.state_dict(),
    }, path)


def load_condition_A_checkpoint(path, load_optimizer=True):
    if not os.path.exists(path):
        print(f"No checkpoint found at {path}, starting from scratch")
        return
    checkpoint = torch.load(path, map_location=device)
    mask_gen.load_state_dict(checkpoint["mask_gen_state_dict"])
    cnn_A.load_state_dict(checkpoint["cnn_A_state_dict"])
    if load_optimizer:
        mask_optimizer.load_state_dict(checkpoint["mask_optimizer_state_dict"])
        cnn_A_optimizer.load_state_dict(checkpoint["cnn_A_optimizer_state_dict"])


def save_condition_B_checkpoint(path):
    torch.save({
        "cnn_B_state_dict": cnn_B.state_dict(),
        "cnn_B_optimizer_state_dict": cnn_B_optimizer.state_dict(),
    }, path)


def load_condition_B_checkpoint(path, load_optimizer=True):
    if not os.path.exists(path):
        print(f"No checkpoint found at {path}, starting from scratch")
        return
    checkpoint = torch.load(path, map_location=device)
    cnn_B.load_state_dict(checkpoint["cnn_B_state_dict"])
    if load_optimizer:
        cnn_B_optimizer.load_state_dict(checkpoint["cnn_B_optimizer_state_dict"])


try:
    load_condition_A_checkpoint(PATH / "SelfShapingMaskCNN.pth", load_optimizer=False)
    load_condition_B_checkpoint(PATH / "PyramidCNN.pth", load_optimizer=False)
except (FileNotFoundError, RuntimeError, KeyError):
    print("Starting from scratch")

## The combined training loop

`Trainer.train()` cannot be reused: it knows nothing about `mask_gen`, and (per this repo's ablation convention) both conditions should train from **one shared atmosphere/noise draw per step** rather than each redrawing independently. `train_combined` below draws `batch = dataset[0]` once per outer step and runs both closed loops against the exact same `opd_gt`/`pupilGT`/`photons`/`ron`/`gain`/`leak`, mirroring `Trainer.train()`'s body (including its per-outer-step reset of the leaky-integrator state to zero) but duplicated for two independent conditions.

**`MaskUpdatePeriod` (`k`) unifies joint and alternating training.** On steps where `u % k == 0`: `mask_gen(wfs_free)` is called *with gradients enabled* (a fresh graph connecting `mask_gen`'s parameters through `wfs_free.mask` to `cnn_A`'s output to `loss_A`), and after the single `total_loss_A.backward()` call, *both* `mask_optimizer` and `cnn_A_optimizer` step from that one shared backward pass -- true joint training. Immediately after, `wfs_free.mask` is detached so it can be safely reused as a frozen constant on the steps in between without ever trying to backward through an already-freed graph. `k = 1` therefore makes every step a true joint update (closest to "simultaneous" this framework gets); `k > 1` holds the mask fixed for `k - 1` steps while `cnn_A` alone adapts to it, genuine coordinate-descent, then both take one joint step together again.

Condition B trains every step, ordinarily -- one tick, one backward pass, one `cnn_B_optimizer` step, computed under `torch.no_grad()` for the WFS/`FramePreprocess` part exactly like `Trainer.train()` does (`wfs_pyramid` has no trainable parameters, so nothing needs gradient through it).

Two oracle-bound trackers (`ideal_loss_A`/`ideal_loss_B`, each condition's own loss evaluated with the reconstructor's output replaced by the oracle correction `Ze`, the same no-grad trick `Trainer.train()` already uses) are kept separately, since A and B's residual-OPD trajectories can diverge once `ClosedLoopIterations > 1`.

In [ ]:
def train_combined(training_steps, closed_loop_iterations, mask_update_period, display=False):
    """display=True live-updates a single figure showing the self-shaping
    mask's current geometry, refreshed every 50th actual mask update (not
    every outer step -- with MaskUpdatePeriod=1 that would refresh every
    step) via IPython.display.clear_output/display. The same imshow() image
    is mutated in place (fig=/axes= update mode) rather than opening a new
    figure per snapshot, so the notebook shows one evolving image instead of
    a stack of dozens of static ones."""
    assert mask_update_period >= 1, "MaskUpdatePeriod must be >= 1"

    cnn_A.train()
    cnn_B.train()

    loss_tracker_A = torch.zeros(training_steps, device=device)
    loss_tracker_B = torch.zeros(training_steps, device=device)
    loss_tracker_ideal_A = torch.zeros(training_steps, device=device)
    loss_tracker_ideal_B = torch.zeros(training_steps, device=device)

    mask_update_count = 0
    display_fig, display_axes = None, None
    if display:
        plt.figure()
        display_fig, display_axes = imshow(mask_gen.phaseMask.detach(), colorbar=True)
        display_axes[0].set_title("Self-shaping mask")
        plt.close(display_fig)  # re-displayed manually below instead of auto-shown inline

    progressBar = tqdm(range(training_steps))

    for u in progressBar:
        do_mask_step = (u % mask_update_period == 0)

        with torch.no_grad():
            batch = dataset[0]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]
            gain = batch["loop_gain"]
            leak = batch["loop_leak"]
            photons = batch["nphotons"]
            ron = batch["ron"]

            modes_gt = torch.matmul(opd_gt.flatten(start_dim=-2), z_inv)

            wfs_free.SetPhotonsAndRON(photons, ron)
            wfs_pyramid.SetPhotonsAndRON(photons, ron)

            z_estimated_A = torch.zeros_like(modes_gt)
            z_buffer_A = torch.zeros_like(modes_gt)
            z_output_A = torch.zeros_like(modes_gt)
            opd_reconstructed_A = torch.zeros_like(opd_gt)

            z_estimated_B = torch.zeros_like(modes_gt)
            z_buffer_B = torch.zeros_like(modes_gt)
            z_output_B = torch.zeros_like(modes_gt)
            opd_reconstructed_B = torch.zeros_like(opd_gt)

        total_loss_A = 0
        total_loss_B = 0
        ideal_loss_A = 0
        ideal_loss_B = 0

        for i in range(closed_loop_iterations):
            with torch.no_grad():
                if i > 0:
                    batch = dataset[i]
                    opd_gt = batch["opd"]
                    pupilGT = batch["pupil"]

                residual_opd_A = opd_gt - opd_reconstructed_A
                residual_opd_B = opd_gt - opd_reconstructed_B

                Ze_A = torch.matmul(residual_opd_A.flatten(start_dim=-2), z_inv)
                Ze_B = torch.matmul(residual_opd_B.flatten(start_dim=-2), z_inv)

                z_estimated_A = z_estimated_A * leak + gain * z_buffer_A
                z_estimated_B = z_estimated_B * leak + gain * z_buffer_B
                z_buffer_A = torch.clone(z_output_A)
                z_buffer_B = torch.clone(z_output_B)

            # ---- Condition A: mask (only on mask-update steps) + CNN ----
            if do_mask_step:
                mask_gen(wfs_free)  # fresh graph: mask_gen params -> wfs_free.mask
            wfs_frames_A = wfs_free(residual_opd_A, pupilGT)
            preprocessed_A = normalize_frame(wfs_frames_A, normalization_A).unsqueeze(1)
            z_output_A = cnn_A(preprocessed_A)

            opd_reconstructed_iter_A = dm(z_output_A @ M2C_T)
            corrected_residual_opd_A = residual_opd_A - opd_reconstructed_iter_A
            total_loss_A = total_loss_A + loss_A(
                Ze_A, z_output_A, pupilGT, residual_opd_A, corrected_residual_opd_A, wfs_frames_A
            ) / closed_loop_iterations

            # ---- Condition B: Pyramid + CNN, trained every step ----
            with torch.no_grad():
                wfs_frames_B = wfs_pyramid(residual_opd_B, pupilGT)
                preprocessed_B = framePreprocessor.ProcessFrame(wfs_frames_B)
            z_output_B = cnn_B(preprocessed_B)

            opd_reconstructed_iter_B = dm(z_output_B @ M2C_T)
            corrected_residual_opd_B = residual_opd_B - opd_reconstructed_iter_B
            total_loss_B = total_loss_B + loss_B(
                Ze_B, z_output_B, pupilGT, residual_opd_B, corrected_residual_opd_B, wfs_frames_B
            ) / closed_loop_iterations

            with torch.no_grad():
                opd_reconstructed_iter_ideal_A = dm(Ze_A @ M2C_T)
                ideal_corrected_residual_opd_A = residual_opd_A - opd_reconstructed_iter_ideal_A
                ideal_loss_A = ideal_loss_A + loss_A(
                    Ze_A, Ze_A, pupilGT, residual_opd_A, ideal_corrected_residual_opd_A, wfs_frames_A
                ) / closed_loop_iterations

                opd_reconstructed_iter_ideal_B = dm(Ze_B @ M2C_T)
                ideal_corrected_residual_opd_B = residual_opd_B - opd_reconstructed_iter_ideal_B
                ideal_loss_B = ideal_loss_B + loss_B(
                    Ze_B, Ze_B, pupilGT, residual_opd_B, ideal_corrected_residual_opd_B, wfs_frames_B
                ) / closed_loop_iterations

                opd_reconstructed_A = dm(z_estimated_A @ M2C_T)
                opd_reconstructed_B = dm(z_estimated_B @ M2C_T)

        # ---- Backprop: condition B, ordinary single-optimizer update, every step ----
        cnn_B_optimizer.zero_grad(set_to_none=True)
        total_loss_B.backward()
        cnn_B_optimizer.step()

        # ---- Backprop: condition A -- joint (mask+CNN) on mask-update steps, CNN-only otherwise ----
        cnn_A_optimizer.zero_grad(set_to_none=True)
        if do_mask_step:
            mask_optimizer.zero_grad(set_to_none=True)
        total_loss_A.backward()
        cnn_A_optimizer.step()
        if do_mask_step:
            mask_optimizer.step()
            wfs_free.mask = wfs_free.mask.detach()
            mask_update_count += 1

            if display and mask_update_count % 50 == 0:
                imshow(mask_gen.phaseMask.detach(), fig=display_fig, axes=display_axes)
                display_axes[0].set_title(f"Self-shaping mask -- step {u} (update #{mask_update_count})")
                clear_output(wait=True)
                ipy_display(display_fig)

        loss_tracker_A[u] = total_loss_A.detach()
        loss_tracker_B[u] = total_loss_B.detach()
        loss_tracker_ideal_A[u] = ideal_loss_A.detach()
        loss_tracker_ideal_B[u] = ideal_loss_B.detach()

        if u % 300 == 1:
            lower = max(0, u - 100)
            progressBar.set_postfix({
                'Loss_A': float(loss_tracker_A[lower:u].mean()),
                'Loss_B': float(loss_tracker_B[lower:u].mean()),
            })

    return loss_tracker_A, loss_tracker_B, loss_tracker_ideal_A, loss_tracker_ideal_B

## Training

`TrainParams['MaskUpdatePeriod']` is set to `1` in the params file (every step is a true joint mask+CNN update). Re-run this cell with a larger `TrainParams['TrainRunNb']` to train further without losing either optimizer's momentum state. Pass `display=True` to watch the self-shaping mask's geometry evolve in place (one `imshow` refreshed via `IPython.display.clear_output`/`display` every 50th actual mask update, not a new figure each time) -- useful the first time you run this, expensive to leave on for a long unattended run.

In [ ]:
loss_tracker_A, loss_tracker_B, loss_tracker_ideal_A, loss_tracker_ideal_B = train_combined(
    TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'], TrainParams['MaskUpdatePeriod'], display=True
)

**A caveat before plotting:** conditions A and B do *not* share the same loss composition (`loss_A` includes `WFSSignalLoss`/`CenterOfGravityLoss` terms `loss_B` doesn't, and `loss_B` includes `Physics_loss`, which `loss_A` doesn't), so their total-loss curves are not on a directly comparable physical scale -- only each condition's own curve against its own oracle bound is meaningful here. The real apples-to-apples comparison is the closed-loop nm-RMS section further down.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(smooth(loss_tracker_A), label="Condition A: self-shaping mask + CNN")
axes[0].plot(smooth(loss_tracker_ideal_A), label="Oracle bound (A)", linestyle=":", color="black")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Loss_A"); axes[0].legend()

axes[1].plot(smooth(loss_tracker_B), label="Condition B: Pyramid + CNN", color="tab:orange")
axes[1].plot(smooth(loss_tracker_ideal_B), label="Oracle bound (B)", linestyle=":", color="black")
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Loss_B"); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
save_condition_A_checkpoint(PATH / "SelfShapingMaskCNN.pth")
save_condition_B_checkpoint(PATH / "PyramidCNN.pth")

## Evaluation loops

Hand-rolled, no-grad closed-loop rollouts mirroring `Trainer.evaluate()` -- none of the three conditions can reuse it as-is: A has no `Trainer` at all, B's would need one built anyway, and C has no reconstructor object, only `GetReconstructedOPD`.

In [ ]:
@torch.no_grad()
def evaluate_A(n_steps, dataset, gain=None, leak=None, psf_sampling=4, psf_fov=20):
    """No-grad closed-loop rollout for condition A (self-shaping mask + CNN)."""
    cnn_A.eval()

    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = gain if gain is not None else batch["loop_gain"]
    leak = leak if leak is not None else batch["loop_leak"]
    photons = batch["nphotons"]
    ron = batch["ron"]

    wfs_free.SetPhotonsAndRON(photons, ron)

    z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
    z_buffer = torch.zeros_like(z_estimated)
    z_output = torch.zeros_like(z_estimated)
    opd_reconstructed = torch.zeros_like(opd_gt)

    opds, pupils, reconstructed, residuals, frames, psfs, outputs = [], [], [], [], [], [], []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed

        wfs_frames = wfs_free(residual_opd, pupilGT)
        psf = wfs_free.GetPSF(residual_opd, pupilGT, psf_sampling, psf_fov)
        preprocessed = normalize_frame(wfs_frames, normalization_A).unsqueeze(1)
        z_output = cnn_A(preprocessed)

        z_buffer = torch.clone(z_output)

        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_buffer

        opd_reconstructed = dm(z_estimated @ M2C_T)

        opds.append(opd_gt); pupils.append(pupilGT); reconstructed.append(opd_reconstructed)
        residuals.append(residual_opd); frames.append(wfs_frames); psfs.append(psf); outputs.append(z_output)

    return EvaluationResult(
        opd=torch.stack(opds), pupil=torch.stack(pupils), opd_reconstructed=torch.stack(reconstructed),
        residual_opd=torch.stack(residuals), wfs_frames=torch.stack(frames), psfs=torch.stack(psfs),
        z_output=torch.stack(outputs),
    )


@torch.no_grad()
def evaluate_B(n_steps, dataset, gain=None, leak=None, psf_sampling=4, psf_fov=20):
    """No-grad closed-loop rollout for condition B (Pyramid + CNN)."""
    cnn_B.eval()

    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = gain if gain is not None else batch["loop_gain"]
    leak = leak if leak is not None else batch["loop_leak"]
    photons = batch["nphotons"]
    ron = batch["ron"]

    wfs_pyramid.SetPhotonsAndRON(photons, ron)

    z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
    z_buffer = torch.zeros_like(z_estimated)
    z_output = torch.zeros_like(z_estimated)
    opd_reconstructed = torch.zeros_like(opd_gt)

    opds, pupils, reconstructed, residuals, frames, psfs, outputs = [], [], [], [], [], [], []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed

        wfs_frames = wfs_pyramid(residual_opd, pupilGT)
        psf = wfs_pyramid.GetPSF(residual_opd, pupilGT, psf_sampling, psf_fov)
        preprocessed = framePreprocessor.ProcessFrame(wfs_frames, False)
        z_output = cnn_B(preprocessed)

        z_buffer = torch.clone(z_output)

        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_buffer

        opd_reconstructed = dm(z_estimated @ M2C_T)

        opds.append(opd_gt); pupils.append(pupilGT); reconstructed.append(opd_reconstructed)
        residuals.append(residual_opd); frames.append(wfs_frames); psfs.append(psf); outputs.append(z_output)

    return EvaluationResult(
        opd=torch.stack(opds), pupil=torch.stack(pupils), opd_reconstructed=torch.stack(reconstructed),
        residual_opd=torch.stack(residuals), wfs_frames=torch.stack(frames), psfs=torch.stack(psfs),
        z_output=torch.stack(outputs),
    )


@torch.no_grad()
def evaluate_C(n_steps, dataset, gain=None, leak=None, psf_sampling=4, psf_fov=20):
    """No-grad closed-loop rollout for condition C (Pyramid + frozen linear reconstructor)."""
    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = gain if gain is not None else batch["loop_gain"]
    leak = leak if leak is not None else batch["loop_leak"]
    photons = batch["nphotons"]
    ron = batch["ron"]

    wfs_pyramid.SetPhotonsAndRON(photons, ron)

    z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
    z_buffer = torch.zeros_like(z_estimated)
    z_output = torch.zeros_like(z_estimated)
    opd_reconstructed = torch.zeros_like(opd_gt)

    opds, pupils, reconstructed, residuals, frames, psfs, outputs = [], [], [], [], [], [], []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed

        wfs_frames = wfs_pyramid(residual_opd, pupilGT)
        psf = wfs_pyramid.GetPSF(residual_opd, pupilGT, psf_sampling, psf_fov)
        z_output = wfs_pyramid.GetReconstructedOPD(wfs_frames)

        z_buffer = torch.clone(z_output)

        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_buffer

        opd_reconstructed = dm(z_estimated @ M2C_T)

        opds.append(opd_gt); pupils.append(pupilGT); reconstructed.append(opd_reconstructed)
        residuals.append(residual_opd); frames.append(wfs_frames); psfs.append(psf); outputs.append(z_output)

    return EvaluationResult(
        opd=torch.stack(opds), pupil=torch.stack(pupils), opd_reconstructed=torch.stack(reconstructed),
        residual_opd=torch.stack(residuals), wfs_frames=torch.stack(frames), psfs=torch.stack(psfs),
        z_output=torch.stack(outputs),
    )

## Three-way closed-loop comparison

Reseed immediately before each condition's rollout (`torch.manual_seed(TrainParams['Seed'])`, this repo's "reseed right before each rollout" convention) so all three see identical wind/r0/noise draws, then compute the steady-state residual wavefront error in nm RMS, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()`/every eval function above already use internally).

In [ ]:
eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

results = {}
nm_rms = {}

torch.manual_seed(TrainParams['Seed'])
results["A: Self-shaping mask + CNN"] = evaluate_A(TrainParams['TestRunNb'], eval_dataset)

torch.manual_seed(TrainParams['Seed'])
results["B: Pyramid + CNN"] = evaluate_B(TrainParams['TestRunNb'], eval_dataset)

torch.manual_seed(TrainParams['Seed'])
results["C: Pyramid + Linear"] = evaluate_C(TrainParams['TestRunNb'], eval_dataset)


def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


for name, result in results.items():
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:28s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")

**A vs B is not a perfectly fair comparison even with matched training.** Condition A's CNN operates on a full raw frame with no pupil-registration prior at all; condition B's benefits from `PyramidWFS`'s known facet geometry via `FramePreprocess`. A gap between them is at least partly a statement about how much that geometric prior is worth, not only about the learned mask's quality.

In [ ]:
param_counts = {
    "A: Self-shaping mask + CNN": total_params_mask + total_params_A,
    "B: Pyramid + CNN": total_params_B,
    "C: Pyramid + Linear": 0,
}

names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(names, rms_values)
ax.set_ylabel("Steady-state residual (nm RMS)")
ax.set_title("Closed-loop performance")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

for name in names:
    print(f"{name:28s} -- {param_counts[name]:>8,} trainable params")

## The learned mask geometry, next to the fixed Pyramid mask

In [ ]:
imshow_multiple(
    [
        {"tensor": mask_gen.phaseMask.detach(), "title": "Learned self-shaping mask (rad)", "colorbar": True},
        {"tensor": wfs_pyramid.phaseMask.detach(), "title": "Fixed Pyramid mask (rad)", "colorbar": True},
    ]
)
plt.show()

## Closed-loop rollout of the winning condition

In [ ]:
best_name = min(nm_rms, key=nm_rms.get)
print(f"Best condition: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

result = results["A: Self-shaping mask + CNN"]
n_frames = result.opd.shape[0]

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())